# Broca Zero — Predictive Coding

**No encoder. No decoder. Just a predictor that maintains a state.**

```
Process each character:
    pred    = W_pred @ state              # what do I expect?
    error   = char_embed[actual] - pred   # how surprised am I?
    state   = normalize(state + α·error)  # update belief

After N characters → state IS the thought vector
    ↓
EAM read → enriched thought ("I've been here before")
    ↓
Generate: predict → sample → surprise → update → repeat
```

**Learned params**: `char_embed` (65×D) + `W_pred` (D×D) = ~25K

**Learning rule**: Hebbian. `ΔW ∝ error ⊗ state`. Local, no backprop.

**Knowledge**: EAM on disk. Swap the EAM, swap the knowledge.

In [ ]:
!pip install -q torch numpy tqdm

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import urllib.request
import json
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Data

In [ ]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
data_path = Path('shakespeare.txt')
if not data_path.exists():
    urllib.request.urlretrieve(url, data_path)

text = data_path.read_text()
print(f'Corpus: {len(text):,} characters')

class CharVocab:
    def __init__(self, text):
        chars = sorted(set(text))
        self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(chars)}
        self.vocab_size = len(chars)

    def encode(self, s):
        return [self.char_to_idx[c] for c in s]

    def decode(self, indices):
        return ''.join(self.idx_to_char[i] for i in indices)

vocab = CharVocab(text)
print(f'Vocab: {vocab.vocab_size} characters')

split = int(0.9 * len(text))
train_text = text[:split]
val_text = text[split:]
print(f'Train: {len(train_text):,} | Val: {len(val_text):,}')

## 2. Predictive Coder

One state vector, updated character by character.

- `char_embed` (vocab × D): learned character representations
- `W_pred` (D × D): transforms state into predicted next-char embedding
- `α`: state update rate (how fast beliefs shift)

The state after processing a sequence IS the thought vector.
There is no encoder. There is no decoder. There is only prediction.

In [ ]:
class PredictiveCoder:
    """Predictive coding state machine.

    Processes characters one at a time:
        pred  = W_pred @ state
        error = char_embed[x] - pred
        state = normalize(state + alpha * error)

    The accumulated state IS the thought vector.
    """

    def __init__(self, vocab_size, dim, alpha=0.1, device='cpu'):
        self.vocab_size = vocab_size
        self.dim = dim
        self.alpha = alpha
        self.device = device

        # Character embeddings — learned
        self.char_embed = torch.randn(vocab_size, dim, device=device)
        self.char_embed = F.normalize(self.char_embed, dim=1)

        # Prediction matrix — learned (Hebbian)
        self.W_pred = torch.randn(dim, dim, device=device) * 0.01

    def initial_state(self, batch_size=1):
        """Start with zero state (no expectations)."""
        return torch.zeros(batch_size, self.dim, device=self.device)

    def step(self, state, char_idx):
        """Process one character.

        Returns: (new_state, error, logits)
            logits are the prediction BEFORE seeing this char
            (i.e., logits predict this char from previous state)
        """
        # Predict: what do I expect?
        pred = state @ self.W_pred.t()                    # (batch, dim)
        logits = pred @ self.char_embed.t()                # (batch, vocab)

        # Observe: what actually happened
        actual = self.char_embed[char_idx]                 # (batch, dim)

        # Surprise
        error = actual - pred                              # (batch, dim)

        # Update belief
        new_state = state + self.alpha * error
        norms = new_state.norm(dim=1, keepdim=True).clamp(min=1e-8)
        new_state = new_state / norms

        return new_state, error, logits

    def to(self, device):
        self.device = device
        self.char_embed = self.char_embed.to(device)
        self.W_pred = self.W_pred.to(device)
        return self

## 3. EAM

In [ ]:
class EAMLayer:
    """Elastic Associative Memory — stores prediction states."""

    def __init__(self, dim, num_locations=2000, k=20, beta=5.0, eta=0.001,
                 device='cpu'):
        self.dim = dim
        self.num_locations = num_locations
        self.k = k
        self.beta = beta
        self.eta = eta
        self.device = device

        self.addresses = torch.randn(num_locations, dim, device=device)
        self.addresses = F.normalize(self.addresses, dim=1)
        self.counters = torch.zeros(num_locations, dim, device=device)
        self.write_counts = torch.zeros(num_locations, device=device)

    @torch.no_grad()
    def write(self, vectors):
        vectors = F.normalize(vectors, dim=1)
        k = min(self.k, self.num_locations)
        sims = torch.mm(vectors, self.addresses.t())
        topk_sims, topk_idx = torch.topk(sims, k, dim=1)
        weights = F.softmax(topk_sims * self.beta, dim=1)

        flat_idx = topk_idx.reshape(-1)
        flat_weights = weights.reshape(-1)
        vecs_expanded = vectors.unsqueeze(1).expand(-1, k, -1).reshape(-1, self.dim)
        weighted_vecs = flat_weights.unsqueeze(1) * vecs_expanded
        idx_for_counters = flat_idx.unsqueeze(1).expand(-1, self.dim)
        self.counters.scatter_add_(0, idx_for_counters, weighted_vecs)
        self.write_counts.scatter_add_(0, flat_idx, flat_weights)

        winners = topk_idx[:, 0]
        diff = vectors - self.addresses[winners]
        self.addresses[winners] += self.eta * diff
        self.addresses[winners] = F.normalize(self.addresses[winners], dim=1)

    @torch.no_grad()
    def read(self, query, max_iter=10):
        """Iterative Hopfield read."""
        query = F.normalize(query, dim=1)
        k = min(self.k, self.num_locations)
        norms = self.counters.norm(dim=1, keepdim=True).clamp(min=1e-8)
        normalized = self.counters / norms

        result = query.clone()
        for _ in range(max_iter):
            sims = torch.mm(result, self.addresses.t())
            topk_sims, topk_idx = torch.topk(sims, k, dim=1)
            weights = F.softmax(topk_sims * self.beta, dim=1)
            gathered = normalized[topk_idx]
            new_result = (weights.unsqueeze(2) * gathered).sum(dim=1)
            new_result = F.normalize(new_result, dim=1)
            if torch.allclose(result, new_result, atol=1e-6):
                break
            result = new_result
        return result

    def reset(self):
        self.addresses = torch.randn_like(self.addresses)
        self.addresses = F.normalize(self.addresses, dim=1)
        self.counters.zero_()
        self.write_counts.zero_()

    def to(self, device):
        self.device = device
        self.addresses = self.addresses.to(device)
        self.counters = self.counters.to(device)
        self.write_counts = self.write_counts.to(device)
        return self

## 4. Configuration

In [ ]:
DIM            = 128     # state dimension (matches HeatherDB default)
CONTEXT_LENGTH = 64      # chars to process before thought vector
CHUNK_SIZE     = 32      # chars to generate after EAM enrichment
ALPHA          = 0.1     # state update rate

# Training
NUM_EPOCHS     = 10
LR             = 0.001   # Hebbian learning rate
BATCH_SIZE     = 256
STRIDE         = 4       # stride through corpus

# EAM (Phase 2)
NUM_LOCATIONS  = 2000

params = vocab.vocab_size * DIM + DIM * DIM
print(f'State dimension: {DIM}')
print(f'Context: {CONTEXT_LENGTH} chars | Generate: {CHUNK_SIZE} chars')
print(f'Learned params: {params:,} = char_embed({vocab.vocab_size}x{DIM}) + W_pred({DIM}x{DIM})')
print(f'Alpha (state update rate): {ALPHA}')
print(f'EAM: {NUM_LOCATIONS} locations x {DIM}d')

## 5. Phase 1 — Learn to Predict (no EAM)

Train the predictive coder on next-character prediction.

At each timestep:
```
pred  = W_pred @ state                    # predict
error = char_embed[actual] - pred         # surprise
state = normalize(state + α·error)        # update

ΔW_pred        ∝ error ⊗ state            # Hebbian
Δchar_embed[x] ∝ error                    # pull toward target
```

No BPTT. Each timestep learns independently.

In [ ]:
coder = PredictiveCoder(vocab.vocab_size, DIM, alpha=ALPHA, device=device)

# Prepare data
train_data = torch.tensor(vocab.encode(train_text), dtype=torch.long, device=device)
val_data = torch.tensor(vocab.encode(val_text), dtype=torch.long, device=device)

train_windows = train_data.unfold(0, CONTEXT_LENGTH, STRIDE)
val_windows = val_data.unfold(0, CONTEXT_LENGTH, 1)[:5000]

n_train = train_windows.shape[0]
n_val = val_windows.shape[0]
print(f'Train sequences: {n_train:,} (stride {STRIDE})')
print(f'Val sequences: {n_val:,}')


def train_epoch(coder, windows, lr):
    """One epoch of Hebbian predictive coding."""
    n = windows.shape[0]
    perm = torch.randperm(n, device=device)
    total_correct = 0
    total_chars = 0
    total_err = 0.0
    n_steps = 0

    for start in range(0, n, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n)
        batch = windows[perm[start:end]]
        bs, seq_len = batch.shape

        state = coder.initial_state(bs)

        # Accumulate Hebbian updates over the sequence
        dW = torch.zeros_like(coder.W_pred)
        d_embed = torch.zeros_like(coder.char_embed)
        embed_counts = torch.zeros(coder.vocab_size, device=device)

        for t in range(seq_len):
            chars = batch[:, t]
            prev_state = state.clone()
            state, error, logits = coder.step(state, chars)

            total_err += error.norm(dim=1).mean().item()
            n_steps += 1

            if t > 0:
                total_correct += (logits.argmax(dim=1) == chars).sum().item()
                total_chars += bs

            # Hebbian: ΔW_pred ∝ error.T @ state_before
            dW += error.t() @ prev_state / bs

            # Δchar_embed[x] ∝ error
            idx = chars.unsqueeze(1).expand(-1, coder.dim)
            d_embed.scatter_add_(0, idx, error)
            embed_counts.scatter_add_(0, chars,
                                      torch.ones(bs, device=device))

        # Apply updates
        coder.W_pred += lr * dW / seq_len

        mask = embed_counts > 0
        d_embed[mask] /= embed_counts[mask].unsqueeze(1)
        coder.char_embed += lr * d_embed
        coder.char_embed = F.normalize(coder.char_embed, dim=1)

    avg_err = total_err / n_steps if n_steps > 0 else 0
    acc = total_correct / total_chars if total_chars > 0 else 0
    return avg_err, acc


def eval_prediction(coder, windows):
    """Evaluate next-char prediction accuracy."""
    correct = 0
    total = 0
    total_err = 0.0
    n_steps = 0

    with torch.no_grad():
        for start in range(0, windows.shape[0], BATCH_SIZE):
            end = min(start + BATCH_SIZE, windows.shape[0])
            batch = windows[start:end]
            bs, seq_len = batch.shape
            state = coder.initial_state(bs)

            for t in range(seq_len):
                chars = batch[:, t]
                state, error, logits = coder.step(state, chars)
                total_err += error.norm(dim=1).mean().item()
                n_steps += 1
                if t > 0:
                    correct += (logits.argmax(dim=1) == chars).sum().item()
                    total += bs

    return total_err / n_steps, correct / total if total > 0 else 0


# ── Phase 1 Training ──
print(f'\n{"Epoch":>5} | {"Train Err":>9} | {"Train Acc":>9} | {"Val Err":>8} | {"Val Acc":>7}')
print('-' * 55)

for epoch in range(NUM_EPOCHS):
    tr_err, tr_acc = train_epoch(coder, train_windows, LR)
    va_err, va_acc = eval_prediction(coder, val_windows)
    print(f'{epoch+1:5d} | {tr_err:9.4f} | {tr_acc:8.1%} | {va_err:8.4f} | {va_acc:6.1%}')

print(f'\nRandom baseline: {1/vocab.vocab_size:.1%}')
print(f'W_pred norm: {coder.W_pred.norm():.3f}')

## 6. Learned Representations

In [ ]:
sims = coder.char_embed @ coder.char_embed.t()
sims_np = sims.cpu().numpy()

pairs = []
for i in range(vocab.vocab_size):
    for j in range(i + 1, vocab.vocab_size):
        pairs.append((sims_np[i, j], vocab.idx_to_char[i], vocab.idx_to_char[j]))
pairs.sort(reverse=True)

print('Most similar characters (learned by prediction):')
for sim, a, b in pairs[:15]:
    a_d = repr(a) if a in ' \n\t' else a
    b_d = repr(b) if b in ' \n\t' else b
    print(f'  {a_d:>5} ~ {b_d:<5} sim={sim:.3f}')

print(f'\nMost dissimilar:')
for sim, a, b in pairs[-5:]:
    a_d = repr(a) if a in ' \n\t' else a
    b_d = repr(b) if b in ' \n\t' else b
    print(f'  {a_d:>5} ~ {b_d:<5} sim={sim:.3f}')

## 7. Phase 2 — Populate EAM

Freeze the predictor. Process the corpus char by char, collecting thought vectors (prediction states) at each context boundary. Write them to the EAM.

The EAM stores prediction states — not encoded text.

In [ ]:
eam = EAMLayer(DIM, num_locations=NUM_LOCATIONS, k=20, beta=5.0, eta=0.001,
               device=device)

TOTAL_LENGTH = CONTEXT_LENGTH + CHUNK_SIZE
full_windows = train_data.unfold(0, TOTAL_LENGTH, 1)
n_full = full_windows.shape[0]
print(f'Writing {n_full:,} thought vectors to EAM...')

with torch.no_grad():
    for start in tqdm(range(0, n_full, BATCH_SIZE), desc='Phase 2'):
        end = min(start + BATCH_SIZE, n_full)
        batch = full_windows[start:end]

        # Process context char by char → thought vector
        state = coder.initial_state(batch.shape[0])
        for t in range(CONTEXT_LENGTH):
            state, _, _ = coder.step(state, batch[:, t])

        # state IS the thought — write it
        eam.write(state)

active = (eam.write_counts > 0).sum().item()
print(f'\nActive locations: {active}/{NUM_LOCATIONS}')
print(f'Mean writes/location: {eam.write_counts[eam.write_counts > 0].mean():.0f}')

## 8. Generate

1. Process prompt char by char → prediction state (thought vector)
2. EAM read → enriched state
3. Generate: predict → sample → surprise → update state → repeat

In [ ]:
def generate(coder, eam, vocab, prompt, num_chars=300, temperature=0.8,
             use_eam=True):
    chars = vocab.encode(prompt)
    generated = list(chars)

    with torch.no_grad():
        # Process prompt char by char
        state = coder.initial_state(1)
        for c in chars:
            idx = torch.tensor([c], device=device)
            state, _, _ = coder.step(state, idx)

        # EAM enrichment
        if use_eam:
            state = eam.read(state)

        # Generate: predict → sample → surprise → update
        for i in range(num_chars):
            pred = state @ coder.W_pred.t()
            logits = pred @ coder.char_embed.t() / temperature
            probs = F.softmax(logits, dim=1)
            next_char = torch.multinomial(probs, 1).squeeze(1)

            generated.append(next_char.item())

            # Update state with surprise from own generation
            state, _, _ = coder.step(state, next_char)

            # Periodically re-query EAM
            if use_eam and (i + 1) % CONTEXT_LENGTH == 0:
                state = eam.read(state)

    return vocab.decode(generated)


prompts = ['ROMEO:', 'To be, or not', 'First Citizen:', 'The king']

for prompt in prompts:
    if not all(c in vocab.char_to_idx for c in prompt):
        continue
    print(f'\n{"="*60}')
    print(f'Prompt: {prompt!r}')
    print(f'{"="*60}')
    print(f'\n[Without EAM — pure predictor]')
    print(generate(coder, eam, vocab, prompt, num_chars=200, use_eam=False))
    print(f'\n[With EAM — enriched prediction]')
    print(generate(coder, eam, vocab, prompt, num_chars=200, use_eam=True))
    print()

## 9. Measure EAM Impact

Does memory enrichment actually improve prediction?

In [ ]:
val_full = val_data.unfold(0, TOTAL_LENGTH, 1)[:3000]

results = {'no_eam': [0, 0], 'eam': [0, 0]}   # [correct, total]

with torch.no_grad():
    for start in tqdm(range(0, len(val_full), BATCH_SIZE), desc='Evaluating'):
        end = min(start + BATCH_SIZE, len(val_full))
        batch = val_full[start:end]
        bs = batch.shape[0]

        # Process context
        state = coder.initial_state(bs)
        for t in range(CONTEXT_LENGTH):
            state, _, _ = coder.step(state, batch[:, t])

        # Without EAM
        s_no = state.clone()
        for t in range(CONTEXT_LENGTH, TOTAL_LENGTH):
            s_no, _, logits = coder.step(s_no, batch[:, t])
            results['no_eam'][0] += (logits.argmax(dim=1) == batch[:, t]).sum().item()
            results['no_eam'][1] += bs

        # With EAM
        s_eam = eam.read(state)
        for t in range(CONTEXT_LENGTH, TOTAL_LENGTH):
            s_eam, _, logits = coder.step(s_eam, batch[:, t])
            results['eam'][0] += (logits.argmax(dim=1) == batch[:, t]).sum().item()
            results['eam'][1] += bs

acc_no = results['no_eam'][0] / results['no_eam'][1]
acc_eam = results['eam'][0] / results['eam'][1]

print(f'\nNext-char accuracy on chunk region (chars {CONTEXT_LENGTH}-{TOTAL_LENGTH}):')
print(f'  Without EAM: {acc_no:.1%}')
print(f'  With EAM:    {acc_eam:.1%}')
print(f'  Lift:        {acc_eam - acc_no:+.1%}')
print(f'  Random:      {1/vocab.vocab_size:.1%}')

## 10. Export

In [ ]:
def export_predictive(coder, eam, vocab, path):
    active_mask = eam.write_counts > 0
    locations = []
    for i in range(eam.num_locations):
        if active_mask[i]:
            locations.append({
                'id': int(i),
                'counter': eam.counters[i].cpu().numpy().astype(np.float64).tolist(),
                'address': eam.addresses[i].cpu().numpy().astype(np.float64).tolist(),
                'write_count': float(eam.write_counts[i].item()),
            })

    export = {
        'config': {
            'dim': coder.dim,
            'num_locations': eam.num_locations,
            'k': eam.k,
            'beta': eam.beta,
        },
        'locations': locations,
        'vocab': {
            'char_to_idx': vocab.char_to_idx,
            'idx_to_char': {str(k): v for k, v in vocab.idx_to_char.items()},
        },
        'model': {
            'type': 'predictive_coding',
            'dim': coder.dim,
            'alpha': coder.alpha,
            'context_length': CONTEXT_LENGTH,
            'chunk_size': CHUNK_SIZE,
        },
    }

    path = Path(path)
    with open(path, 'w') as f:
        json.dump(export, f)

    weights_path = path.with_name(path.stem + '_predictor.pt')
    torch.save({
        'char_embed': coder.char_embed.cpu(),
        'W_pred': coder.W_pred.cpu(),
    }, weights_path)

    print(f'Exported {len(locations)} EAM locations to {path}')
    print(f'Predictor saved to {weights_path}')
    sz = path.stat().st_size / 1024 / 1024
    sz_w = weights_path.stat().st_size / 1024
    print(f'Size: {sz:.1f} MB (EAM) + {sz_w:.0f} KB (predictor)')

export_predictive(coder, eam, vocab, 'broca_zero.json')

## Architecture

```
R → state_1    (big surprise — first char, no context)
O → state_2    (moderate)
M → state_3    (expected after RO)
E → state_4    (expected)
O → state_5    (expected — ROMEO is common)
: → state_6    (expected after a name)
  → state_7    (expected)
O → state_8    (slight surprise)
...             (state evolves through prediction errors)
h → state_N    ← THIS is the thought vector
                        ↓
              EAM Read (enriched by past experience)
                        ↓
              Generate from enriched state
```

| Component | Size | Role |
|---|---|---|
| char_embed | 65 × 128 = 8,320 | Character representations |
| W_pred | 128 × 128 = 16,384 | State → predicted embedding |
| EAM | 2,000 × 128 (disk) | Prediction state memory |
| **Total learned** | **24,704** | |

Learning: `ΔW ∝ error ⊗ state` (Hebbian, local, no backprop)

Knowledge surgery: swap the EAM. Same predictor, different knowledge.